In [1]:
import json
from collections.abc import Sequence
import requests
from bs4 import BeautifulSoup

from langchain_community.utilities import DuckDuckGoSearchAPIWrapper as ddg
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_ollama import ChatOllama

In [2]:
class LocalLLM:
    def __init__(
        self,
        model: str='gemma4:e4b',
        temperature: float=0.7,
    ):
        self._model = model
        self._temperature = temperature
        
    def __call__(self):
        return ChatOllama(
            model=self._model,
            temperature=self._temperature,
            validate_model_on_init=True,
        )

In [3]:
class WebSearch:
    def __init__(self, timeout:int = 15):
        self._timeout = timeout
        self._headers = {
            'User-Agent': (
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/124.0.0.0 Safari/537.36'
            ),
            'Accept-Language': 'en-US,en;q=0.9',
        }    

    def search(
        self,
        web_query: str,
        num_results: int,
    ) -> Sequence[str]:
        return [
            result['link'] for result in ddg().results(
                web_query, num_results
            )
        ]
        
    def scrape(self, url: str) -> str:
        """Scrape a webpage."""
        try:
            response = requests.get(url, headers=self._headers, timeout=self._timeout)
            if response.status_code == 200:
                return BeautifulSoup(
                    response.text, 'html.parser'
                ).get_text(separator=' ', strip=True)
            else:
                return f"Failed to retrieve the webpage: Status code {response.status_code}"
        except Exception as e:
            print(e)
            return f"Failed to retrieve the webpage: {e}"
            

In [4]:
llm = LocalLLM()
ws = WebSearch()

In [5]:
class Assistant:
    def __init__(self, llm: LocalLLM):
        self.template = PromptTemplate.from_template(template=self._read_template())
        self._llm = llm

    def _read_template(self):
        return '''
        You are skilled at assigning a research question to the correct research assistant. 
        There are various research assistants available, each specialized in an area of expertise. 
        Each assistant is identified by a specific type. Each assistant has specific instructions to undertake the research.
        
        How to select the correct assistant:
        You must select the relevant assistant depending on the topic of the question, which should match the area of expertise of the assistant.
        
        
        Here are some examples on how to return the correct assistant information, depending on the question asked:
        
        **Examples:**
        
        Question: "Should I invest in Apple stocks?"
        Response: 
        {{
            "assistant_type": "Financial analyst assistant",
            "assistant_instructions": "You are a seasoned finance analyst AI assistant. Your primary goal is to compose comprehensive, astute, impartial, and methodically arranged financial reports based on provided data and trends.",
            "user_question": {user_question}
        }}
        
        Question: "what are the most interesting sites in Tel Aviv?"
        Response: 
        {{
            "assistant_type": "Tour guide assistant",
            "assistant_instructions": "You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.",
            "user_question": "{user_question}"
        }}
        
        
        Question: "Is Messi a good soccer player?"
        Response: 
        {{
            "assistant_type": "Sport expert assistant",
            "assistant_instructions": "You are an experienced AI sport assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured sport reports on given sport personalities, or sport events, including factual details, statistics and insights.",
            "user_question": "{user_question}"
        }}
        
        
        Now that you have understood all the above, select the correct reserach assistant for the following question.
        Question: {user_question}
        Response:
        '''

    def __call__(self):
        return (
            {'user_question': RunnablePassthrough()} 
            | self.template | self._llm() | JsonOutputParser()
        )    

In [6]:
# Test Assistant()
question = 'What can I see and do in the Spanish town of Astorga?'
assistant = Assistant(llm)
assistant_instructions_dict = assistant().invoke(question)
print(assistant_instructions_dict)

{'assistant_type': 'Tour guide assistant', 'assistant_instructions': 'You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}


In [7]:
class ExpandSearch:
    def __init__(self, llm: LocalLLM, num_search_queries: int = 3):
        self._llm = llm
        self._num_search_queries = num_search_queries
        self.template = PromptTemplate.from_template(template=self._read_template())

    def _read_template(self) -> str:
        return '''
        You construct useful questions based on user input.
        Instructions:
        {assistant_instructions}
        
        Write {num_search_queries} web search queries to gather as much information as possible 
        on the following question: {user_question}. Your objective is to write a report based on the information you find.
        You must respond with a list of queries such as query1, query2, query3 in the following format: 
        [
            {{"search_query": "query1", "user_question": "{user_question}" }},
            {{"search_query": "query2", "user_question": "{user_question}" }},
            {{"search_query": "query3", "user_question": "{user_question}" }}
        ]
        '''

    def __call__(self):
        return (
            RunnableLambda(
                lambda x: {
                    'assistant_instructions': x['assistant_instructions'],
                    'num_search_queries': self._num_search_queries,
                    'user_question': x['user_question']
                }
            )
            | self.template | self._llm() | JsonOutputParser()
        )

In [8]:
#Test ExpandSearch()
assistant_instruction_dict = {
    'assistant_type': 'Tour guide assistant',
    'assistant_instructions': 'You are a world-travelled AI tour guide assistant. Your main purpose is to draft engaging, insightful, unbiased, and well-structured travel reports on given locations, including history, attractions, and cultural insights.',
    'user_question': 'What can I see and do in the Spanish town of Astorga?'
}

add_queries = ExpandSearch(llm)
web_searches_list = add_queries().invoke(assistant_instruction_dict)
print(f'Input: {assistant_instruction_dict['user_question']}')
print('Expanded queries:')
for query in web_searches_list:
    print(query['search_query'])

Input: What can I see and do in the Spanish town of Astorga?
Expanded queries:
Must-see attractions and things to do in Astorga, Spain
History and architectural highlights of Astorga Spain
Local activities, gastronomy, and day trips from Astorga


In [9]:
class RetrieveUrls:
    def __init__(self, ws, num_search_results_per_query: int = 3):
        self._ws = ws
        self._num_search_results_per_query = num_search_results_per_query        
        
    def __call__(self):
        return (
            RunnableLambda(
                lambda x: [
                    {
                        'result_url': url, 
                        'search_query': x['search_query'],
                        'user_question': x['user_question']
                    }
                    for url in self._ws.search(
                        web_query=x['search_query'], 
                        num_results=self._num_search_results_per_query
                    )
                ]
            )
        )

In [10]:
#Test RetrieveUrls()
web_search_dict = {"search_query": "Astorga Spain attractions", "user_question": "What can I see and do in the Spanish town of Astorga?"}
retriever = RetrieveUrls(ws)
result_urls_list = retriever().invoke(web_search_dict)
print(result_urls_list)

[{'result_url': 'https://www.spain.info/en/destination/astorga/', 'search_query': 'Astorga Spain attractions', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'result_url': 'https://astorga.co/en/what-is-the-best-time-to-visit-astorga-in-2025/', 'search_query': 'Astorga Spain attractions', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'result_url': 'https://astorga.co/en/holy-week-in-spain-2024/', 'search_query': 'Astorga Spain attractions', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}]


In [11]:
class Summarizer:
    """Summarize a webpage."""
    def __init__(self, llm, ws, max_text: int = 10000):
        self._llm = llm
        self._ws = ws
        self._max_text = max_text
        self._template = PromptTemplate.from_template(template=self._read_template())

    def _read_template(self) -> str:
        return '''
        You are an intelligent summarizer.
        Read the following text:
        Text: {search_result_text} 
        
        
        Using the above text, answer in short the following question.
        Question: {search_query}
         
        If you cannot answer the question above using the text provided above, then just summarize the text. 
        Include all factual information, numbers, stats etc if available.        
        '''
        
    def __call__(self):        
        return (
            RunnableLambda(lambda x: {
                    'search_result_text': self._ws.scrape(url=x['result_url'])[:self._max_text],
                    'result_url': x['result_url'], 
                    'search_query': x['search_query'],
                    'user_question': x['user_question'],
                })
            
            | RunnableParallel ({
                    'text_summary': self._template | self._llm() | StrOutputParser(),
                    'result_url': lambda x: x['result_url'],
                    'user_question': lambda x: x['user_question']            
                })
            
            | RunnableLambda(lambda x: {
                    'summary': f"Source Url: {x['result_url']}\nSummary: {x['text_summary']}",
                    'user_question': x['user_question']
                }) 
        )
                

In [12]:
#Test Summarizer()
result_url_dict = {"result_url": "https://citiesandattractions.com/spain/astorga-spain-uncovering-the-jewels-of-a-hidden-spanish-gem/", "search_query": "Astorga Spain attractions", "user_question": "What can I see and do in the Spanish town of Astorga?"}
summarize = Summarizer(llm, ws)
search_text_summary = summarize().invoke(result_url_dict)
print(search_text_summary)

{'summary': 'Source Url: https://citiesandattractions.com/spain/astorga-spain-uncovering-the-jewels-of-a-hidden-spanish-gem/\nSummary: Astorga, Spain, is a destination rich in history, culture, and nature. Key attractions include:\n\n*   **Architecture & Museums:**\n    *   **Episcopal Palace:** An impressive museum designed by Antoni Gaudi.\n    *   **Cathedral of Santa Maria de Astorga:** Built in the 15th century.\n    *   **Roman Walls and Museum:** The walls date back to the 3rd century AD, and the museum exhibits ancient Roman artifacts.\n    *   **Chocolate Factory Museum:** Offers guided tours on chocolate making.\n    *   **Palace of Gaudi:** A small palace designed by Antoni Gaudi, serving as a cultural center.\n*   **Nature & Cuisine:**\n    *   **Sierra de los Ancares:** A nearby mountain range offering stunning views and hiking trails.\n    *   **Wineries:** The region is known for sampling delicious local wines.\n    *   **Local Delicacies:** Famous for "Cocido Maragato,"

In [13]:
class Researcher:
    def __init__(self, llm):
        self._llm = llm
        self._template = PromptTemplate.from_template(template=self._read_template())
        self._ws = WebSearch()        
        self._assistant = Assistant(self._llm)
        self._add_queries = ExpandSearch(self._llm)
        self._retriever = RetrieveUrls(self._ws)
        self._summarizer = Summarizer(self._llm, self._ws)        
        self._search_and_summarization_chain = self._search_and_summarize()

    def _read_template(self) -> str:
        return '''
        You are an AI critical thinker research assistant.
        Your sole purpose is to write well written, critically acclaimed, objective and structured reports on given text.
        
        Information: 
        {research_summary}
        
        Using the above information, answer the following question or topic: "{user_question}" in a detailed report.
        The report should focus on the answer to the question, should be well structured, informative,
        in depth, with facts and numbers if available and a minimum of 1,200 words.
        
        You should strive to write the report as long as you can using all relevant and necessary information provided.
        You must write the report with markdown syntax.
        You MUST determine your own concrete and valid opinion based on the given information. Do NOT deter to general and meaningless conclusions.
        Write all used source urls at the end of the report, and make sure to not add duplicated sources, but only one reference for each.
        You must write the report in apa format.        
        '''
    
    def _search_and_summarize(self):
        return (
            self._retriever()
            | self._summarizer().map()
            | RunnableLambda(lambda x: {
                'summary': '\n'.join([i['summary'] for i in x]), 
                'user_question': x[0]['user_question'] if len(x) > 0 else ''
            })
        )

    def execute(self):
        """Execute Research Summarizer."""
        return (
            self._assistant()
            | self._add_queries()
            | self._search_and_summarization_chain.map()
            | RunnableLambda(lambda x: {
                'research_summary': '\n\n'.join([i['summary'] for i in x]),
                'user_question': x[0]['user_question'] if len(x) > 0 else ''
            })
            | self._template | self._llm() | StrOutputParser()
        )

In [14]:
question = 'What can I see and do in the Spanish town of Astorga?'
researcher = Researcher(llm)
web_research_report = researcher.execute().invoke(question)

In [15]:
print(web_research_report)

# The Convergence of Empire, Faith, and Artistry: A Comprehensive Guide to Astorga, Spain

***

## Introduction

Astorga, situated within the region of León, Castile and León, Spain, transcends the definition of a mere historical town; it is a profound cultural nexus. As a destination, Astorga offers an unparalleled journey through multiple epochs, successfully blending the monumental grandeur of Roman antiquity with the whimsical artistic genius of Antoni Gaudí, all while remaining at the spiritual crossroads of major European pilgrimage routes. The city's identity is defined by this extraordinary convergence: a confluence of Roman infrastructure, the deep spiritual tradition of the Camino de Santiago, and a unique, celebrated heritage in artisan chocolate.

To explore Astorga is to embark on a multidisciplinary study of Spanish history, where the threads of the empire, the spiritual quest, and the industrial revolution meet in palpable architectural and cultural forms. This report pr